In [0]:
from datetime import date

current_year = date.today().year
years_list = [str(x) for x in range(2006, current_year + 1)]

dbutils.widgets.dropdown(
    label="01 - Start Year",
    name="start_year",
    defaultValue=years_list[0],
    choices=years_list
)

dbutils.widgets.dropdown(
    label="02 - End Year",
    name="end_year",
    defaultValue=years_list[-1],
    choices=years_list
)


In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze.bcb_sgs_indexes (
    index_date DATE NOT NULL,
    yearly_selic DOUBLE,
    monthly_ipca DOUBLE,
    daily_usd DOUBLE
)
USING DELTA
CLUSTER BY (index_date)

In [0]:
import pyspark.sql.functions as F
import pandas as pd
from bcb import sgs

In [0]:
# ----------------------------
# 1. Baixar dados (pandas)
# ----------------------------
start_date = dbutils.widgets.get("start_year") + "-01-01"
end_date = dbutils.widgets.get("end_year") + "-12-31"

dados_pd = sgs.get(
    {
        'SELIC': 432,  # Selic % a.a.
        'IPCA': 433,   # IPCA mensal %
        'USD/BRL': 1   # Taxa Cambial USD x BRL
    },
    start=start_date, end=end_date
)

dados_pd.index = pd.to_datetime(dados_pd.index)
dados_pd = dados_pd.reset_index()

# ----------------------------
# 2. Converter para Spark
# ----------------------------
df_sgs = spark.createDataFrame(dados_pd)

# Garantir tipo correto
df_sgs = (
    df_sgs
        .withColumn("data", F.to_date("Date"))
        .withColumnsRenamed(
            {
                'data': 'index_date',
                'SELIC': 'yearly_selic',
                'IPCA': 'monthly_ipca',
                'USD/BRL': 'daily_usd'
            }
        )
        .drop("Date")
        .orderBy('index_date')
)

In [0]:
(
    df_sgs.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("bronze.bcb_sgs_indexes")
)